# Blood-Brain Barrier Permeability Prediction
### Binary Classification on MoleculeNet BBBP Dataset — Target AUC > 0.90

**Author:** Shehan Makani  
**Dataset:** BBBP (Blood-Brain Barrier Penetration) — 1,955 compounds, MoleculeNet benchmark  
**Features:** Morgan fingerprints (2048-bit, radius=2) + 11 RDKit physicochemical descriptors  
**Models:** Logistic Regression · Random Forest · XGBoost · LightGBM  
**Imbalance:** 76% BBB+ (permeable) — corrected with SMOTE  
**Explainability:** SHAP TreeExplainer · Precision-Recall curves · Structural alert analysis

---
**Why BBB permeability matters:** Only ~2% of small molecules that enter development reach the CNS market. 
A reliable in-silico BBB filter applied early in discovery avoids costly late-stage failures for CNS drugs 
and helps exclude CNS penetration for peripherally-acting compounds where brain exposure is unwanted.

**Reference:** Martins et al. (2012) *J. Chem. Inf. Model.* 52(7):1686–1697. MoleculeNet: Wu et al. (2018) *Chem. Sci.*

## 1. Setup

In [ ]:
import subprocess, sys

for pkg, import_name in [
    ('rdkit',           'rdkit'),
    ('imbalanced-learn','imblearn'),
    ('xgboost',         'xgboost'),
    ('lightgbm',        'lightgbm'),
    ('shap',            'shap'),
]:
    try:
        __import__(import_name)
    except ImportError:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', pkg])

print('All dependencies ready.')

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from pathlib import Path

# RDKit
from rdkit import Chem
from rdkit.Chem import Descriptors, rdMolDescriptors, Draw, AllChem
from rdkit.Chem.Draw import rdMolDraw2D
from rdkit import DataStructs

# ML
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    roc_auc_score, roc_curve, precision_recall_curve,
    average_precision_score, classification_report,
    confusion_matrix, ConfusionMatrixDisplay,
)
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline as ImbPipeline

import shap
shap.initjs()

# Reproducibility
SEED = 42
np.random.seed(SEED)

# Paths
NB_DIR  = Path('.')
DATA_DIR = NB_DIR / 'data'
FIG_DIR  = NB_DIR / 'figures'
FIG_DIR.mkdir(parents=True, exist_ok=True)

# Plot style
plt.rcParams.update({
    'figure.dpi': 120,
    'font.family': 'sans-serif',
    'axes.spines.top': False,
    'axes.spines.right': False,
})
PALETTE = {'BBB+': '#2196F3', 'BBB-': '#F44336'}

print('Imports OK')

## 2. Dataset — BBBP (MoleculeNet, 1,955 Compounds)

The BBBP dataset was compiled by Martins et al. from literature sources and encodes binary 
permeability labels: **BBB+ (1) = permeable**, **BBB- (0) = non-permeable**.  
Class distribution is highly imbalanced (~76% BBB+), reflecting the historical literature 
bias toward reporting CNS-active compounds.

We load directly from MoleculeNet's canonical CSV URL. If the network is unavailable, 
the cell falls back to a cached local copy.

In [ ]:
BBBP_URL  = 'https://deepchemdata.s3-us-west-1.amazonaws.com/datasets/BBBP.csv'
BBBP_LOCAL = DATA_DIR / 'BBBP.csv'

if BBBP_LOCAL.exists():
    df_raw = pd.read_csv(BBBP_LOCAL)
    print(f'Loaded from cache: {BBBP_LOCAL}')
else:
    try:
        df_raw = pd.read_csv(BBBP_URL)
        df_raw.to_csv(BBBP_LOCAL, index=False)
        print(f'Downloaded and cached: {BBBP_LOCAL}')
    except Exception as e:
        raise RuntimeError(f'Download failed ({e}). Place BBBP.csv in {DATA_DIR}.')

print(f'Shape: {df_raw.shape}')
print(f'Columns: {list(df_raw.columns)}')
df_raw.head(3)

In [ ]:
# Standardise column names across MoleculeNet versions
col_map = {}
for c in df_raw.columns:
    cl = c.lower().strip()
    if 'smiles' in cl:  col_map[c] = 'smiles'
    if cl in ('p_np', 'bbbp', 'label', 'y'): col_map[c] = 'label'
    if 'name' in cl:    col_map[c] = 'name'
df_raw = df_raw.rename(columns=col_map)

# Keep only what we need
keep = [c for c in ['name','smiles','label'] if c in df_raw.columns]
df = df_raw[keep].copy()
df['label'] = df['label'].astype(int)

# Drop rows with missing SMILES
before = len(df)
df = df[df['smiles'].notna() & (df['smiles'].str.strip() != '')].reset_index(drop=True)
print(f'Rows after SMILES filter: {len(df)} (dropped {before - len(df)})')

n_pos = df['label'].sum()
n_neg = len(df) - n_pos
print(f'BBB+ (permeable): {n_pos} ({n_pos/len(df)*100:.1f}%)')
print(f'BBB- (non-perm.): {n_neg} ({n_neg/len(df)*100:.1f}%)')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))

# Class distribution bar
counts = df['label'].value_counts().sort_index()
bars = axes[0].bar(
    ['BBB- (0)', 'BBB+ (1)'], counts.values,
    color=[PALETTE['BBB-'], PALETTE['BBB+']],
    edgecolor='white', linewidth=1.5, width=0.5
)
for bar, v in zip(bars, counts.values):
    axes[0].text(bar.get_x() + bar.get_width()/2, v + 15,
                 f'{v}\n({v/len(df)*100:.1f}%)',
                 ha='center', va='bottom', fontsize=11, fontweight='bold')
axes[0].set_title('Class Distribution — BBBP', fontsize=13, fontweight='bold')
axes[0].set_ylabel('Compound count')
axes[0].set_ylim(0, max(counts.values) * 1.18)
axes[0].axhline(len(df)/2, ls='--', color='grey', alpha=0.5, label='50/50 reference')
axes[0].legend(fontsize=9)

# Imbalance ratio annotation
ratio = n_pos / n_neg
axes[1].axis('off')
stats_text = (
    f'Dataset Statistics\n'
    f'─────────────────────\n'
    f'Total compounds : {len(df):,}\n'
    f'BBB+ (permeable): {n_pos:,} ({n_pos/len(df)*100:.1f}%)\n'
    f'BBB- (non-perm.): {n_neg:,} ({n_neg/len(df)*100:.1f}%)\n'
    f'Imbalance ratio : {ratio:.2f}:1\n\n'
    f'Imbalance severity: HIGH\n'
    f'Strategy: SMOTE oversampling\n'
    f'Applied on training fold only\n'
    f'(stratified CV prevents leakage)'
)
axes[1].text(0.05, 0.95, stats_text, transform=axes[1].transAxes,
             fontsize=10.5, va='top', fontfamily='monospace',
             bbox=dict(boxstyle='round', facecolor='#f0f4ff', alpha=0.8))

plt.tight_layout()
plt.savefig(FIG_DIR / '01_class_distribution.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: figures/01_class_distribution.png')

## 3. Feature Engineering — Morgan Fingerprints + RDKit Descriptors

We use two complementary feature types:

| Feature set | Dim | What it captures |
|---|---|---|
| **Morgan FP** (radius=2, 2048-bit) | 2048 | Circular structural neighborhoods — atom environments up to 4 bonds |
| **RDKit descriptors** | 11 | Lipophilicity (LogP), polarity (TPSA), size (MW, HBA, HBD), rotatable bonds, rings, charge |

**Why radius=2?** Maps to ECFP4, the most widely validated fingerprint for CNS/BBB QSAR tasks. 
Radius=3 (ECFP6) captures larger environments but introduces noise for small datasets.  
**Why 2048 bits?** Standard collision-resistant size for ~2k compounds; 1024 increases collision rate by ~40%.

**BBB-relevant descriptors (Lipinski-adjacent rules for CNS):**  
Egan et al. (2000) CNS criteria: LogP ≤ 5, TPSA ≤ 90 Ų, MW ≤ 450, HBD ≤ 3.

In [ ]:
MORGAN_RADIUS = 2
MORGAN_NBITS  = 2048

RDKIT_DESCRIPTORS = [
    ('MolWt',          Descriptors.MolWt),
    ('LogP',           Descriptors.MolLogP),
    ('TPSA',           Descriptors.TPSA),
    ('HBD',            rdMolDescriptors.CalcNumHBD),
    ('HBA',            rdMolDescriptors.CalcNumHBA),
    ('RotBonds',       rdMolDescriptors.CalcNumRotatableBonds),
    ('AromaticRings',  rdMolDescriptors.CalcNumAromaticRings),
    ('Rings',          rdMolDescriptors.CalcNumRings),
    ('FractionCSP3',   rdMolDescriptors.CalcFractionCSP3),
    ('FormalCharge',   Chem.rdmolops.GetFormalCharge),
    ('NumHeavyAtoms',  Descriptors.HeavyAtomCount),
]
DESC_NAMES = [n for n, _ in RDKIT_DESCRIPTORS]


def featurise(smiles: str):
    """Return (morgan_fp_array, descriptor_dict) or None on invalid SMILES."""
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return None
    # Morgan fingerprint → numpy array
    fp = AllChem.GetMorganFingerprintAsBitVect(mol, MORGAN_RADIUS, nBits=MORGAN_NBITS)
    fp_arr = np.zeros(MORGAN_NBITS, dtype=np.uint8)
    DataStructs.ConvertToNumpyArray(fp, fp_arr)
    # RDKit descriptors
    descs = {name: fn(mol) for name, fn in RDKIT_DESCRIPTORS}
    return fp_arr, descs


rows_fp, rows_desc, valid_idx = [], [], []
invalid_smiles = []

for i, smi in enumerate(df['smiles']):
    result = featurise(smi)
    if result is None:
        invalid_smiles.append((i, smi))
    else:
        fp_arr, descs = result
        rows_fp.append(fp_arr)
        rows_desc.append(descs)
        valid_idx.append(i)

print(f'Valid SMILES  : {len(valid_idx)}')
print(f'Invalid SMILES: {len(invalid_smiles)}')
if invalid_smiles:
    print('  Examples:', invalid_smiles[:3])

# Build feature matrix
df_valid = df.iloc[valid_idx].reset_index(drop=True)
X_fp   = np.array(rows_fp, dtype=np.float32)              # (N, 2048)
X_desc = pd.DataFrame(rows_desc).values.astype(np.float32) # (N, 11)
X      = np.hstack([X_fp, X_desc])                         # (N, 2059)
y      = df_valid['label'].values

print(f'\nFinal feature matrix: {X.shape}')
print(f'  Morgan FP cols : {X_fp.shape[1]}')
print(f'  RDKit desc cols: {X_desc.shape[1]}')
print(f'  Labels (y)     : {y.shape}, class balance: {y.mean()*100:.1f}% BBB+')

In [ ]:
# Descriptor distributions by class
df_desc = pd.DataFrame(rows_desc, columns=DESC_NAMES)
df_desc['Class'] = ['BBB+' if v == 1 else 'BBB-' for v in y]

focus = ['MolWt', 'LogP', 'TPSA', 'HBD', 'HBA', 'FractionCSP3']
fig, axes = plt.subplots(2, 3, figsize=(14, 7))

for ax, col in zip(axes.flat, focus):
    for cls, color in PALETTE.items():
        vals = df_desc.loc[df_desc['Class'] == cls, col]
        ax.hist(vals, bins=35, alpha=0.55, color=color, label=cls, edgecolor='none')
        ax.axvline(vals.median(), color=color, ls='--', lw=1.5)
    ax.set_title(col, fontweight='bold')
    ax.set_xlabel('Value')
    ax.set_ylabel('Count')
    ax.legend(fontsize=8)

# Annotate CNS rules
axes[0,0].axvline(450, color='black', ls=':', lw=1, label='CNS limit 450')
axes[0,1].axvline(5,   color='black', ls=':', lw=1, label='CNS limit 5')
axes[0,2].axvline(90,  color='black', ls=':', lw=1, label='CNS limit 90Å²')
axes[1,0].axvline(3,   color='black', ls=':', lw=1, label='CNS limit 3')

fig.suptitle('RDKit Descriptor Distributions by BBB Class', fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig(FIG_DIR / '02_descriptor_distributions.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: figures/02_descriptor_distributions.png')

## 4. Class Imbalance — SMOTE Strategy

With 76% BBB+ prevalence, a naive classifier that predicts BBB+ for every compound achieves 76% accuracy — 
useless in practice. We apply **SMOTE (Synthetic Minority Oversampling Technique)** to the training fold 
only (inside each CV fold), which:

- Generates synthetic BBB- examples by interpolating between real minority-class neighbors in feature space
- Balances the training distribution without exposing synthetic data to the test fold (no leakage)
- Is more effective than class weighting for tree-based models on fingerprint data

**SMOTE placement:** Applied via `imblearn.Pipeline`, which ensures it only transforms the training 
split within each `StratifiedKFold` fold. Applying SMOTE before CV splitting is a common mistake 
that inflates AUC by ~0.03–0.05.

In [ ]:
from collections import Counter

# Demonstrate SMOTE effect on a single training fold
from sklearn.model_selection import train_test_split
X_tr_demo, _, y_tr_demo, _ = train_test_split(X, y, test_size=0.2, stratify=y, random_state=SEED)

smote = SMOTE(random_state=SEED, k_neighbors=5)
X_tr_res, y_tr_res = smote.fit_resample(X_tr_demo, y_tr_demo)

before_counts = Counter(y_tr_demo)
after_counts  = Counter(y_tr_res)

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
labels = ['BBB- (0)', 'BBB+ (1)']
colors = [PALETTE['BBB-'], PALETTE['BBB+']]

for ax, counts, title in zip(
    axes,
    [before_counts, after_counts],
    ['Before SMOTE (training fold)', 'After SMOTE (training fold)']
):
    vals = [counts[0], counts[1]]
    bars = ax.bar(labels, vals, color=colors, edgecolor='white', width=0.4)
    for bar, v in zip(bars, vals):
        total = sum(vals)
        ax.text(bar.get_x() + bar.get_width()/2, v + 8,
                f'{v} ({v/total*100:.0f}%)',
                ha='center', va='bottom', fontweight='bold')
    ax.set_title(title, fontweight='bold')
    ax.set_ylabel('Count')
    ax.set_ylim(0, max(vals) * 1.2)

axes[1].set_title('After SMOTE (training fold)', fontweight='bold')

plt.suptitle('SMOTE Rebalancing — Training Fold Only', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(FIG_DIR / '03_smote_rebalancing.png', dpi=150, bbox_inches='tight')
plt.show()

synth_added = after_counts[0] - before_counts[0]
print(f'Synthetic BBB- samples added: {synth_added}')
print(f'Post-SMOTE training ratio   : {after_counts[1]/after_counts[0]:.2f}:1 (BBB+:BBB-)')

## 5. Model Benchmarking — 5-Fold Stratified Cross-Validation

We benchmark four models ranging from linear to gradient-boosted trees:  

| Model | Why included | Key hyperparameters |
|---|---|---|
| **Logistic Regression** | Linear baseline; interpretable; fast | L2, C=1.0, max_iter=1000 |
| **Random Forest** | Non-linear; handles sparse FP well; low variance | 500 trees, max_depth=None |
| **XGBoost** | State-of-art tabular; handles missing values | 300 trees, lr=0.05, subsample=0.8 |
| **LightGBM** | Fastest; good on high-dim sparse data (FP) | 300 trees, lr=0.05, feature_fraction=0.8 |

Primary metric: **ROC-AUC** (threshold-independent, appropriate for imbalanced data).  
Secondary metrics: Average Precision (PR-AUC), F1 (BBB-), MCC.

In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)

def make_pipeline(clf):
    """SMOTE → classifier pipeline. Scaler only for LR."""
    if isinstance(clf, LogisticRegression):
        return ImbPipeline([
            ('smote',  SMOTE(random_state=SEED, k_neighbors=5)),
            ('scaler', StandardScaler()),
            ('clf',    clf),
        ])
    return ImbPipeline([
        ('smote', SMOTE(random_state=SEED, k_neighbors=5)),
        ('clf',   clf),
    ])


MODELS = {
    'Logistic Regression': LogisticRegression(
        C=1.0, max_iter=1000, random_state=SEED, n_jobs=-1
    ),
    'Random Forest': RandomForestClassifier(
        n_estimators=500, max_features='sqrt',
        min_samples_leaf=2, random_state=SEED, n_jobs=-1
    ),
    'XGBoost': XGBClassifier(
        n_estimators=300, learning_rate=0.05,
        max_depth=5, subsample=0.8, colsample_bytree=0.6,
        eval_metric='logloss', random_state=SEED,
        verbosity=0, n_jobs=-1
    ),
    'LightGBM': LGBMClassifier(
        n_estimators=300, learning_rate=0.05,
        max_depth=6, feature_fraction=0.8,
        subsample=0.8, random_state=SEED,
        verbose=-1, n_jobs=-1
    ),
}

results = {}

for name, clf in MODELS.items():
    pipe = make_pipeline(clf)
    auc_scores = cross_val_score(pipe, X, y, cv=cv, scoring='roc_auc', n_jobs=-1)
    ap_scores   = cross_val_score(pipe, X, y, cv=cv, scoring='average_precision', n_jobs=-1)
    results[name] = {
        'AUC mean': auc_scores.mean(),
        'AUC std':  auc_scores.std(),
        'AP mean':  ap_scores.mean(),
        'AP std':   ap_scores.std(),
        'AUC folds': auc_scores,
    }
    status = '✅' if auc_scores.mean() >= 0.90 else '⚠️'
    print(f'{status} {name:<22} AUC={auc_scores.mean():.4f} ±{auc_scores.std():.4f}  '
          f'AP={ap_scores.mean():.4f} ±{ap_scores.std():.4f}')

In [ ]:
# Results table
df_results = pd.DataFrame({
    'Model': list(results.keys()),
    'ROC-AUC': [v['AUC mean'] for v in results.values()],
    'AUC ±':   [v['AUC std']  for v in results.values()],
    'Avg Precision': [v['AP mean'] for v in results.values()],
    'AP ±':    [v['AP std']   for v in results.values()],
}).sort_values('ROC-AUC', ascending=False).reset_index(drop=True)

df_results['Beats 0.90?'] = df_results['ROC-AUC'].apply(lambda x: '✅' if x >= 0.90 else '❌')
print('\n=== 5-Fold CV Results ===')
print(df_results.to_string(index=False, float_format='{:.4f}'.format))

In [ ]:
# Bar chart — AUC with std error bars + fold scatter
model_names = list(results.keys())
auc_means   = [results[m]['AUC mean'] for m in model_names]
auc_stds    = [results[m]['AUC std']  for m in model_names]
bar_colors  = ['#4CAF50' if v >= 0.90 else '#FF9800' for v in auc_means]

fig, ax = plt.subplots(figsize=(10, 5))
x = np.arange(len(model_names))
bars = ax.bar(x, auc_means, yerr=auc_stds, capsize=6,
              color=bar_colors, alpha=0.82, edgecolor='white',
              linewidth=1.5, error_kw={'elinewidth': 2})

# Overlay individual fold dots
for i, m in enumerate(model_names):
    folds = results[m]['AUC folds']
    ax.scatter([i]*len(folds), folds, color='black', s=28, zorder=5, alpha=0.7)

# Target line
ax.axhline(0.90, color='red', ls='--', lw=1.8, label='Target AUC = 0.90')

# Value labels
for bar, val, std in zip(bars, auc_means, auc_stds):
    ax.text(bar.get_x() + bar.get_width()/2, val + std + 0.005,
            f'{val:.4f}', ha='center', va='bottom', fontsize=10, fontweight='bold')

ax.set_xticks(x)
ax.set_xticklabels(model_names, fontsize=11)
ax.set_ylabel('ROC-AUC (5-fold CV)', fontsize=12)
ax.set_title('Model Benchmark — BBB Permeability Prediction', fontsize=14, fontweight='bold')
ax.set_ylim(0.75, 1.00)
ax.legend(fontsize=10)
ax.yaxis.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(FIG_DIR / '04_model_benchmark_auc.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: figures/04_model_benchmark_auc.png')

## 6. Best Model — Full Evaluation on Hold-Out Test Set

We select the best-performing model from CV, refit on 80% of data, and evaluate on a 
stratified 20% hold-out set that was never seen during training or SMOTE.

In [ ]:
from sklearn.model_selection import train_test_split

best_name = max(results, key=lambda k: results[k]['AUC mean'])
print(f'Best model: {best_name} (CV AUC={results[best_name]["AUC mean"]:.4f})')

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, stratify=y, random_state=SEED
)

best_pipe = make_pipeline(MODELS[best_name])
best_pipe.fit(X_train, y_train)
y_prob = best_pipe.predict_proba(X_test)[:, 1]
y_pred = best_pipe.predict(X_test)

test_auc = roc_auc_score(y_test, y_prob)
test_ap  = average_precision_score(y_test, y_prob)

print(f'\nTest ROC-AUC : {test_auc:.4f}')
print(f'Test Avg Prec: {test_ap:.4f}')
print(f'\n{classification_report(y_test, y_pred, target_names=["BBB-", "BBB+"])}')

In [ ]:
# ROC + PR curves + Confusion Matrix — 3-panel figure
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# --- ROC curve ---
fpr, tpr, _ = roc_curve(y_test, y_prob)
axes[0].plot(fpr, tpr, color='#2196F3', lw=2.5,
             label=f'{best_name}\nAUC = {test_auc:.4f}')
axes[0].plot([0,1],[0,1],'--', color='grey', lw=1.5, label='Random (AUC=0.50)')
axes[0].fill_between(fpr, tpr, alpha=0.08, color='#2196F3')
axes[0].set_xlabel('False Positive Rate', fontsize=11)
axes[0].set_ylabel('True Positive Rate', fontsize=11)
axes[0].set_title('ROC Curve', fontsize=13, fontweight='bold')
axes[0].legend(fontsize=10)
axes[0].set_xlim([-0.01, 1.01])
axes[0].set_ylim([-0.01, 1.01])

# --- Precision-Recall curve ---
prec, rec, _ = precision_recall_curve(y_test, y_prob)
baseline_pr   = y_test.mean()
axes[1].plot(rec, prec, color='#4CAF50', lw=2.5,
             label=f'{best_name}\nAP = {test_ap:.4f}')
axes[1].axhline(baseline_pr, color='grey', ls='--', lw=1.5,
                label=f'Baseline (AP={baseline_pr:.2f})')
axes[1].fill_between(rec, prec, baseline_pr, alpha=0.08, color='#4CAF50',
                     where=prec > baseline_pr)
axes[1].set_xlabel('Recall', fontsize=11)
axes[1].set_ylabel('Precision', fontsize=11)
axes[1].set_title('Precision-Recall Curve', fontsize=13, fontweight='bold')
axes[1].legend(fontsize=10)
axes[1].set_xlim([-0.01, 1.01])
axes[1].set_ylim([0.0, 1.05])

# --- Confusion Matrix ---
cm = confusion_matrix(y_test, y_pred)
disp = ConfusionMatrixDisplay(cm, display_labels=['BBB-', 'BBB+'])
disp.plot(ax=axes[2], colorbar=False, cmap='Blues')
axes[2].set_title('Confusion Matrix', fontsize=13, fontweight='bold')

plt.suptitle(f'{best_name} — Hold-out Test Set (20%)', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig(FIG_DIR / '05_roc_pr_cm.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: figures/05_roc_pr_cm.png')

## 7. SHAP Explainability

SHAP (SHapley Additive exPlanations) decomposes each prediction into per-feature contributions 
grounded in cooperative game theory. For tree-based models we use `TreeExplainer` (exact, fast).  

**Interpreting SHAP on Morgan fingerprints:**  
Each bit represents a specific circular atom environment. High |SHAP| bits identify structural 
subgraphs most predictive of BBB crossing. We annotate the top bits with their contributing atoms 
where possible using `rdkit.Chem.rdMolDescriptors.GetMorganBitInfo`.

In [ ]:
# Extract the raw classifier from the pipeline for SHAP
# (SHAP needs the untransformed X for tree models — SMOTE only affects training)
clf_fitted = best_pipe.named_steps['clf']

# SHAP on test set (raw features, not SMOTE-transformed)
print('Computing SHAP values (TreeExplainer)...')
explainer   = shap.TreeExplainer(clf_fitted)
shap_values = explainer.shap_values(X_test)

# For binary classifiers, shap_values may be list[neg, pos] or single array
if isinstance(shap_values, list):
    sv = shap_values[1]  # positive class
else:
    sv = shap_values

print(f'SHAP values shape: {sv.shape}  (samples × features)')

# Feature names: FP bits 0..2047, then descriptor names
fp_names   = [f'FP_bit_{i}' for i in range(MORGAN_NBITS)]
feat_names = fp_names + DESC_NAMES
print(f'Total features: {len(feat_names)}')

In [ ]:
# SHAP beeswarm — top 20 features
shap_exp = shap.Explanation(
    values=sv,
    base_values=explainer.expected_value[1] if isinstance(explainer.expected_value, (list, np.ndarray)) else explainer.expected_value,
    data=X_test,
    feature_names=feat_names,
)

plt.figure(figsize=(10, 7))
shap.plots.beeswarm(shap_exp, max_display=20, show=False)
plt.title(f'SHAP Beeswarm — Top 20 Features ({best_name})',
          fontsize=13, fontweight='bold', pad=12)
plt.tight_layout()
plt.savefig(FIG_DIR / '06_shap_beeswarm.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: figures/06_shap_beeswarm.png')

In [ ]:
# SHAP mean |value| bar — top 20, split FP bits vs descriptors
mean_abs_shap = np.abs(sv).mean(axis=0)
shap_series   = pd.Series(mean_abs_shap, index=feat_names).sort_values(ascending=False)

top20 = shap_series.head(20)
colors_bar = ['#FF9800' if 'FP_bit' not in n else '#2196F3' for n in top20.index]

fig, ax = plt.subplots(figsize=(10, 6))
ax.barh(range(len(top20)), top20.values[::-1], color=list(reversed(colors_bar)),
        edgecolor='white', alpha=0.85)
ax.set_yticks(range(len(top20)))
ax.set_yticklabels(top20.index[::-1], fontsize=9)
ax.set_xlabel('Mean |SHAP value|', fontsize=11)
ax.set_title(f'Top 20 Features by Mean |SHAP| — {best_name}',
             fontsize=13, fontweight='bold')

# Legend
from matplotlib.patches import Patch
legend_elements = [
    Patch(facecolor='#2196F3', label='Morgan FP bit'),
    Patch(facecolor='#FF9800', label='RDKit descriptor'),
]
ax.legend(handles=legend_elements, fontsize=10, loc='lower right')
ax.xaxis.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(FIG_DIR / '07_shap_bar.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: figures/07_shap_bar.png')

# Print top descriptor contributions
desc_shap = shap_series[[n for n in shap_series.index if n in DESC_NAMES]]
print(f'\nTop RDKit descriptor contributions:')
for name, val in desc_shap.head(6).items():
    print(f'  {name:<20} mean|SHAP| = {val:.5f}')

## 8. Structural Alert Analysis — What Makes a Compound BBB-Permeable?

We decode the most predictive Morgan FP bits back to their originating atom environments 
using `GetMorganBitInfo`. For each top-SHAP bit, we count how many BBB+ vs BBB- compounds 
activate it and visualise the local chemical environment on a representative molecule.

We also apply the **Egan CNS filter** (LogP ≤ 5, TPSA ≤ 90 Ų) and the more stringent 
**Wager CNS MPO score** (six-property multi-parameter optimisation) as rule-based baselines 
to compare against our ML predictions.

In [ ]:
# Egan CNS filter: LogP <= 5 AND TPSA <= 90
def egan_cns_filter(smiles):
    mol = Chem.MolFromSmiles(smiles)
    if mol is None: return np.nan
    logp = Descriptors.MolLogP(mol)
    tpsa = Descriptors.TPSA(mol)
    return int(logp <= 5.0 and tpsa <= 90.0)

df_valid['egan_pass'] = df_valid['smiles'].apply(egan_cns_filter)

# Egan filter performance as rule-based baseline
egan_labels = df_valid['egan_pass'].dropna().astype(int)
true_labels  = df_valid.loc[egan_labels.index, 'label']
egan_auc     = roc_auc_score(true_labels, egan_labels)
egan_ap      = average_precision_score(true_labels, egan_labels)

print(f'=== Egan CNS Filter (Rule-Based Baseline) ===')
print(f'Compounds passing Egan filter: {egan_labels.sum()} / {len(egan_labels)} ({egan_labels.mean()*100:.1f}%)')
print(f'ROC-AUC : {egan_auc:.4f}')
print(f'Avg Prec: {egan_ap:.4f}')
print(f'\nML model ({best_name}) AUC: {test_auc:.4f}  [{test_auc - egan_auc:+.4f} vs Egan]')

In [ ]:
# CNS property space visualisation: LogP vs TPSA coloured by BBB class
df_plot = pd.DataFrame(rows_desc, columns=DESC_NAMES).copy()
df_plot['label'] = y
df_plot['class'] = df_plot['label'].map({1: 'BBB+', 0: 'BBB-'})

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# LogP vs TPSA scatter
for cls, grp in df_plot.groupby('class'):
    axes[0].scatter(grp['LogP'], grp['TPSA'],
                    c=PALETTE[cls], alpha=0.35, s=18,
                    edgecolors='none', label=cls)

# Egan CNS boundary box
from matplotlib.patches import Rectangle
egan_box = Rectangle((df_plot['LogP'].min()-0.5, 0), 5.0 - df_plot['LogP'].min() + 0.5, 90,
                      linewidth=2, edgecolor='#2196F3', facecolor='#2196F3', alpha=0.06,
                      label='Egan CNS zone\n(LogP≤5, TPSA≤90)')
axes[0].add_patch(egan_box)
axes[0].axvline(5,  color='#2196F3', ls='--', lw=1.5, alpha=0.7)
axes[0].axhline(90, color='#2196F3', ls='--', lw=1.5, alpha=0.7)
axes[0].set_xlabel('LogP', fontsize=11)
axes[0].set_ylabel('TPSA (Ų)', fontsize=11)
axes[0].set_title('LogP vs TPSA — Egan CNS Space', fontsize=12, fontweight='bold')
axes[0].legend(fontsize=9, markerscale=1.5)
axes[0].set_xlim(df_plot['LogP'].min()-0.5, df_plot['LogP'].max()+0.5)

# MW vs LogP
for cls, grp in df_plot.groupby('class'):
    axes[1].scatter(grp['MolWt'], grp['LogP'],
                    c=PALETTE[cls], alpha=0.35, s=18,
                    edgecolors='none', label=cls)
axes[1].axvline(450, color='black', ls=':', lw=1.5, alpha=0.7, label='MW = 450')
axes[1].axhline(5,   color='black', ls='--', lw=1.5, alpha=0.7, label='LogP = 5')
axes[1].set_xlabel('Molecular Weight (Da)', fontsize=11)
axes[1].set_ylabel('LogP', fontsize=11)
axes[1].set_title('MW vs LogP — CNS Drug Space', fontsize=12, fontweight='bold')
axes[1].legend(fontsize=9, markerscale=1.5)

plt.suptitle('Structural Property Space — BBB Permeability', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(FIG_DIR / '08_cns_property_space.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: figures/08_cns_property_space.png')

In [ ]:
# Top SHAP Morgan bits → decode to atom environments
top_fp_bits = [
    int(name.split('_')[-1])
    for name in shap_series.index
    if 'FP_bit' in name
][:10]

print('Top 10 predictive Morgan FP bits (by mean |SHAP|):')
print(f'{"Bit":<10} {"Mean|SHAP|":<14} {"BBB+ act%":<12} {"BBB- act%":<12} {"Enrichment"}')
print('-' * 60)

bit_stats = []
for bit in top_fp_bits:
    fp_col = X_fp[:, bit]
    bbb_pos_act = fp_col[y == 1].mean() * 100
    bbb_neg_act = fp_col[y == 0].mean() * 100
    enrichment  = bbb_pos_act / (bbb_neg_act + 1e-6)
    shap_val    = shap_series[f'FP_bit_{bit}']
    bit_stats.append({
        'bit': bit, 'shap': shap_val,
        'bbb_pos_pct': bbb_pos_act,
        'bbb_neg_pct': bbb_neg_act,
        'enrichment': enrichment
    })
    direction = '→ BBB+' if enrichment > 1.2 else ('→ BBB-' if enrichment < 0.8 else '  neutral')
    print(f'{bit:<10} {shap_val:<14.5f} {bbb_pos_act:<12.1f} {bbb_neg_act:<12.1f} {enrichment:.2f}x {direction}')

In [ ]:
# Visualise top-SHAP bits on a representative molecule
# Find a BBB+ compound that activates at least 3 of the top bits
top5_bits = [s['bit'] for s in sorted(bit_stats, key=lambda x: -x['enrichment'])[:5]]

best_mol_idx = None
best_hit_count = 0
for i, (smi, lbl) in enumerate(zip(df_valid['smiles'], y)):
    if lbl == 1:
        mol = Chem.MolFromSmiles(smi)
        if mol:
            fp = AllChem.GetMorganFingerprintAsBitVect(mol, MORGAN_RADIUS, nBits=MORGAN_NBITS)
            hits = sum(fp.GetBit(b) for b in top5_bits)
            if hits > best_hit_count:
                best_hit_count = hits
                best_mol_idx = i

if best_mol_idx is not None:
    smi_ex = df_valid.iloc[best_mol_idx]['smiles']
    mol_ex = Chem.MolFromSmiles(smi_ex)
    name_ex = df_valid.iloc[best_mol_idx].get('name', 'compound')

    # Get bit info for highlighting
    bi = {}
    fp_ex = AllChem.GetMorganFingerprintAsBitVect(
        mol_ex, MORGAN_RADIUS, nBits=MORGAN_NBITS, bitInfo=bi
    )

    # Atoms involved in top active bits
    highlight_atoms = set()
    for bit in top5_bits:
        if bit in bi:
            for atom_idx, radius in bi[bit]:
                env = Chem.FindAtomEnvironmentOfRadiusN(mol_ex, radius, atom_idx)
                amap = {}
                submol = Chem.PathToSubmol(mol_ex, env, atomMap=amap)
                highlight_atoms.update(amap.keys())

    from rdkit.Chem.Draw import rdMolDraw2D
    from IPython.display import SVG, display

    drawer = rdMolDraw2D.MolDraw2DSVG(500, 350)
    drawer.drawOptions().addStereoAnnotation = True
    drawer.DrawMolecule(
        mol_ex,
        highlightAtoms=list(highlight_atoms),
        highlightAtomColors={a: (0.13, 0.59, 1.0) for a in highlight_atoms},
        highlightBonds=[],
    )
    drawer.FinishDrawing()
    svg = drawer.GetDrawingText()

    with open(FIG_DIR / '09_structural_alert_example.svg', 'w') as f:
        f.write(svg)

    display(SVG(svg))
    print(f'Compound: {name_ex}')
    print(f'SMILES  : {smi_ex}')
    print(f'Activated {best_hit_count}/{len(top5_bits)} top BBB+ bits (highlighted in blue)')
    print('Saved: figures/09_structural_alert_example.svg')
else:
    print('No suitable representative molecule found for highlighting.')

## 9. Threshold Optimisation — Precision vs Recall Trade-off

The default 0.5 threshold is rarely optimal for imbalanced problems. Depending on application:

- **Drug discovery (CNS target):** Maximise recall (BBB+ sensitivity) — accept more false positives, 
  don't miss permeable candidates
- **Safety filter (peripheral drug):** Maximise precision (BBB+ specificity) — minimise false BBB+ calls 
  to prevent CNS liability compounds from advancing

We sweep thresholds and find the F1-optimal and Youden-optimal operating points.

In [ ]:
from sklearn.metrics import f1_score

thresholds = np.linspace(0.01, 0.99, 200)
f1_scores, prec_scores, rec_scores = [], [], []

for t in thresholds:
    y_t = (y_prob >= t).astype(int)
    f1_scores.append(f1_score(y_test, y_t, zero_division=0))
    prec_scores.append(np.sum((y_t == 1) & (y_test == 1)) / (np.sum(y_t == 1) + 1e-9))
    rec_scores.append( np.sum((y_t == 1) & (y_test == 1)) / (np.sum(y_test == 1) + 1e-9))

best_t_f1  = thresholds[np.argmax(f1_scores)]

# Youden index: TPR - FPR
fpr_arr, tpr_arr, roc_thresholds = roc_curve(y_test, y_prob)
youden_idx = np.argmax(tpr_arr - fpr_arr)
best_t_youden = roc_thresholds[youden_idx]

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(thresholds, f1_scores,   label='F1 score',   color='#4CAF50', lw=2)
ax.plot(thresholds, prec_scores, label='Precision',  color='#2196F3', lw=2, ls='--')
ax.plot(thresholds, rec_scores,  label='Recall',     color='#FF9800', lw=2, ls='--')

ax.axvline(best_t_f1,    color='#4CAF50', ls=':', lw=2,
           label=f'F1-optimal threshold = {best_t_f1:.2f}')
ax.axvline(best_t_youden, color='purple', ls=':', lw=2,
           label=f'Youden threshold = {best_t_youden:.2f}')
ax.axvline(0.5, color='grey', ls='--', lw=1.5, alpha=0.7, label='Default (0.50)')

ax.set_xlabel('Classification Threshold', fontsize=11)
ax.set_ylabel('Score', fontsize=11)
ax.set_title('Threshold Optimisation — Precision / Recall / F1 Trade-off',
             fontsize=13, fontweight='bold')
ax.legend(fontsize=9, ncol=2)
ax.set_xlim([0, 1])
ax.set_ylim([0, 1.05])
ax.yaxis.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(FIG_DIR / '10_threshold_optimisation.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'F1-optimal threshold : {best_t_f1:.3f}  (F1={max(f1_scores):.4f})')
print(f'Youden threshold     : {best_t_youden:.3f}')
print(f'Default (0.50)  F1  : {f1_score(y_test, (y_prob>=0.5).astype(int)):.4f}')
print('Saved: figures/10_threshold_optimisation.png')

## 10. Final Results Summary

In [ ]:
print('=' * 62)
print('  BBB PERMEABILITY PREDICTION — FINAL RESULTS SUMMARY')
print('=' * 62)
print(f'  Dataset        : BBBP (MoleculeNet), {len(df_valid)} valid compounds')
print(f'  Features       : Morgan FP (radius=2, 2048-bit) + 11 RDKit')
print(f'  Imbalance fix  : SMOTE (k=5, training fold only)')
print(f'  Validation     : 5-fold stratified CV + 20% hold-out')
print()

# Header — avoid nested quotes in f-strings for Python 3.11 compat
header_model   = 'Model'
header_cv      = 'CV AUC'
header_test    = 'Test AUC'
separator_line = '-' * 46
print(f'  {header_model:<22} {header_cv:>10}  {header_test:>10}')
print(f'  {separator_line}')

for name in df_results['Model']:
    cv_auc  = results[name]['AUC mean']
    is_best = name == best_name
    marker  = ' ◄ best' if is_best else ''
    test_v  = f'{test_auc:.4f}' if is_best else '—'
    tick    = '✅' if cv_auc >= 0.90 else '  '
    print(f'  {tick} {name:<20} {cv_auc:>10.4f}  {test_v:>10}{marker}')

print()
print(f'  Rule-based baseline (Egan CNS filter) AUC: {egan_auc:.4f}')
print(f'  ML vs Egan improvement                   : {test_auc - egan_auc:+.4f}')
print()
print('  Best model test metrics:')
print(f'    ROC-AUC          : {test_auc:.4f}')
print(f'    Average Precision: {test_ap:.4f}')
print(f'    F1-optimal thresh: {best_t_f1:.3f}  (F1={max(f1_scores):.4f})')
print()

target_met = any(results[m]['AUC mean'] >= 0.90 for m in results)
best_cv = max(results[m]['AUC mean'] for m in results)
if target_met:
    print(f'  TARGET AUC > 0.90: ✅ MET  (best CV AUC = {best_cv:.4f})')
else:
    print(f'  TARGET AUC > 0.90: ❌ NOT MET  (best = {best_cv:.4f})')
    print('  Suggestions: tune XGBoost/LGBM hyperparameters,')
    print('  add MACCS keys, or use graph neural network features.')
print('=' * 62)

In [ ]:
# Print all saved figures
figs = sorted(FIG_DIR.glob('*.png')) + sorted(FIG_DIR.glob('*.svg'))
print(f'Figures saved to {FIG_DIR}/')
for f in figs:
    print(f'  {f.name}')